# Negative GENIE FSI knob weights

> **Note (2026-10-05):** `create_rw_syst_df.py` now sets negative `*_UBGenie` knob weights to zero when building
> `spline_weights_df.parquet`.  The outputs saved here were made from the earlier, unclipped spline parquet; rerunning
> on a parquet built after that change will show no negative weights.

Some GENIE hadron-transport (FSI) reweighting knobs give **negative per-event weights**, even for events whose
GENIE tune weight is perfectly valid (this is separate from the invalid-tune events that `save_PROfit_rootfiles.py`
now resets to one).  This notebook looks at:

1. which knobs and which knob values (σ points) produce negative weights, and how often;
2. what the per-event weight curves look like, and which events get negative weights;
3. whether it matters for the analysis: PROfit's per-bin spline response, recomputed exactly as PROfit does it,
   compared with the same response with the negative weights clipped at zero.

**How the knobs are stored and used.**  Each `*_UBGenie` FSI knob is a 7-element list per event: the weight at
σ = −3, −2, −1, 0, +1, +2, +3 (element 3 is the CV, σ = 0).  The PROfit XMLs use them as `type="spline"` with
`force_0_cv="true"` and no `knobvals`, so PROfit takes the default knob values −3…+3.  In `PROsyst::FillSpline`,
for every analysis bin and knob value it forms

$$R_\sigma(\text{bin}) = \frac{\sum_{\text{events in bin}} w \, u_\sigma}{\sum_{\text{events in bin}} w}\,,\qquad
\text{then with force\_0\_cv: } R_\sigma \to R_\sigma / R_0,$$

where $w$ is the event's `net_weight` and $u_\sigma$ its knob weight at σ, and interpolates a spline in σ through
those 7 points per bin.  So a negative $u_\sigma$ lowers that bin's response at σ, and in a sparsely populated bin
a few such events could dominate it.

**Summary** (details and the numbers behind it in the conclusions at the bottom):
* Only 4 of the 55 `*_UBGenie` knobs have negative weights for valid-tune events: FrCEx_N (8.5% of events), FrInel_pi and
  FrCEx_pi (~0.8%), FrAbs_pi (0.1%).  None have non-finite weights.
* The negative weights occur only on the + side, at +2σ and +3σ; ±1σ is essentially unaffected.
* Only FrCEx_N at +3σ (and slightly +2σ) changes PROfit's bin responses noticeably, so this matters only if a fit
  pulls FrCEx_N well beyond +1σ.

In [ ]:
import os
import sys

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib as mpl
import xml.etree.ElementTree as ET
import re

sys.path.append(os.path.abspath(".."))
# src/ goes first: save_PROfit_rootfiles uses non-package imports, and this directory has an
# older save_PROfit_rootfiles.py of its own that would otherwise shadow it
sys.path.insert(0, os.path.abspath("../src"))
import save_PROfit_rootfiles as spr
assert os.path.dirname(spr.__file__) == os.path.abspath("../src"), spr.__file__
# some reweighting modules imported through postprocessing call matplotlib.use("Agg"); switch back
%matplotlib inline
from file_locations import intermediate_files_location

mpl.rcParams["figure.dpi"] = 110

TRAINING = spr.DEFAULT_TRAINING
STUDY = spr.STUDIES["open_data"]
XML_PATH = "../profit_xmls/bagels_full_open_data.xml"
SPLINE_PATH = f"{intermediate_files_location}/spline_weights_df.parquet"
KEYS = ["filetype", "run", "subrun", "event"]
SIGMAS = np.array([-3, -2, -1, 0, 1, 2, 3])
CV_INDEX = 3

# the GENIE FSI (hA) knobs: fate fractions and mean free paths
FSI_KNOBS = [f"{p}_{h}_UBGenie" for p in ["FrAbs", "FrCEx", "FrInel"] for h in ["pi", "N"]] + ["MFP_pi_UBGenie", "MFP_N_UBGenie"]
TRUTH_COLS = ["wc_truth_isCC", "wc_truth_nuScatType", "wc_truth_NprimPio", "wc_true_num_prim_protons",
              "wc_true_num_prim_neutrons", "wc_truth_nuEnergy"]
TUNE_COL = spr.GENIE_CV_WEIGHT_COL   # pandora_weightTune
print(f"training {TRAINING}, study '{STUDY['label']}'")

## Load the events PROfit sees

`build_minimal_df` is the same function `save_PROfit_rootfiles.py` uses: the open-data prediction, BDT test events
only (weighted up by 1/frac_test), with `reco_category`.  As in `write_withspline_root`, MC events are inner-joined to
their spline weights and weighted up by the per-filetype `spline_processed_fraction_weight`, and invalid-tune events
get unit GENIE weights.

In [ ]:
minimal = spr.build_minimal_df(TRAINING, STUDY).collect(engine="streaming")
net_weight_col = STUDY["net_weight_col"]
mc = minimal.filter(~pl.col("filetype").is_in(spr.NO_SPLINE_FILETYPES))
fractions = spr.compute_spline_fractions(mc, SPLINE_PATH)
mc = mc.join(fractions, on="filetype", how="left")

truth = (pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet").select(KEYS + TRUTH_COLS + [TUNE_COL])
         .join(mc.lazy().select(KEYS), on=KEYS, how="semi").collect(engine="streaming"))
spline_schema = pl.scan_parquet(SPLINE_PATH).select(FSI_KNOBS).collect_schema()
spline = (pl.scan_parquet(SPLINE_PATH).select(KEYS + FSI_KNOBS)
          .join(mc.lazy().select(KEYS), on=KEYS, how="semi").collect(engine="streaming"))

ev = (mc.join(spline, on=KEYS, how="inner").join(truth, on=KEYS, how="left")
      .with_columns((pl.col(net_weight_col) * pl.col("spline_processed_fraction_weight")).alias("w"))
      .with_columns(spr.unit_genie_weights_where_invalid(spline_schema)))
print(f"{ev.height:,} MC events with spline weights (of {mc.height:,} MC events in the PROfit selection)")
print(ev.group_by("filetype").agg(pl.len().alias("events"), pl.col("w").sum().alias("sum_w")).sort("events", descending=True))

# per-knob (events x 7) weight arrays
U = {k: np.vstack(ev[k].to_numpy()) for k in FSI_KNOBS}
w = ev["w"].to_numpy()

The σ = 0 element of these knob lists is not 1: it is the event's GENIE tune weight (`pandora_weightTune`), so the stored weights are absolute, not relative to the CV.  With `force_0_cv` PROfit's response is therefore $\sum w\,u_\sigma / \sum w\,u_0$, i.e. the relative shift $u_\sigma/u_0$ averaged with weights $w \times$ tune.  This is also why the weight distributions below reach ~12 (and ~30 for FrCEx_N) at every σ.

In [ ]:
tune = ev[TUNE_COL].to_numpy()
valid = ev[spr.GENIE_WEIGHTS_VALID_COL].to_numpy()
for k in FSI_KNOBS:
    u0 = U[k][valid, CV_INDEX]
    print(f"{k:20s} valid-tune events: sigma=0 element == tune weight: {np.mean(np.isclose(u0, tune[valid], rtol=1e-4)):.4f}, "
          f"== 1: {np.mean(np.isclose(u0, 1.0, rtol=1e-4)):.4f}")

## Which GENIE knobs have negative weights?

A scan of all `*_UBGenie` knobs on a random sample of the events above (valid-tune events only, so this is not the
invalid-tune problem).

In [ ]:
all_schema = pl.scan_parquet(SPLINE_PATH).collect_schema()
ub_knobs = [c for c, d in all_schema.items() if isinstance(d, pl.List) and c.endswith("_UBGenie")]
sample_keys = ev.filter(pl.col(spr.GENIE_WEIGHTS_VALID_COL)).select(KEYS).sample(min(300_000, ev.height), seed=0)
scan = (pl.scan_parquet(SPLINE_PATH).select(KEYS + ub_knobs)
        .join(sample_keys.lazy(), on=KEYS, how="semi").collect(engine="streaming"))
rows = []
for k in ub_knobs:
    rows.append(dict(
        knob=k,
        frac_events_negative=scan.select(pl.col(k).list.eval(pl.element() < 0).list.any().mean()).item(),
        frac_events_nonfinite=scan.select(pl.col(k).list.eval(~pl.element().is_finite()).list.any().mean()).item(),
        min_weight=scan.select(pl.col(k).list.min().min()).item(),
    ))
neg_table = pl.DataFrame(rows).filter((pl.col("frac_events_negative") > 0) | (pl.col("frac_events_nonfinite") > 0)).sort("frac_events_negative", descending=True)
print(f"{len(ub_knobs)} UBGenie knobs scanned on {scan.height:,} valid-tune events; knobs with any negative or non-finite weight:")
print(neg_table)

## At which σ points?

In [ ]:
frac_neg = np.array([[np.mean(U[k][:, j] < 0) for j in range(7)] for k in FSI_KNOBS])
fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(np.where(frac_neg > 0, frac_neg, np.nan), cmap="viridis", norm=mpl.colors.LogNorm(vmin=1e-5, vmax=0.2), aspect="auto")
for i in range(len(FSI_KNOBS)):
    for j in range(7):
        ax.text(j, i, f"{100 * frac_neg[i, j]:.2g}%" if frac_neg[i, j] > 0 else "0", ha="center", va="center", fontsize=7,
                color="w" if 0 < frac_neg[i, j] < 0.01 else "k")
ax.set_xticks(range(7), [f"{s:+d}σ" for s in SIGMAS])
ax.set_yticks(range(len(FSI_KNOBS)), [k.replace("_UBGenie", "") for k in FSI_KNOBS])
ax.set_title("Fraction of events with a negative weight, per knob and σ point")
fig.colorbar(im, ax=ax, label="fraction of events")
plt.tight_layout()
plt.show()

## What do the per-event weight curves look like?

Weight vs σ for random events with and without a negative weight somewhere in the curve.  Events whose hadron
underwent the varied fate get a weight that rises with σ; events whose hadrons underwent other fates get a
compensating weight that falls with σ (GENIE keeps the fate fractions summing to one).

In [ ]:
NEG_KNOBS = [k for k in FSI_KNOBS if (U[k] < 0).any()]
rng = np.random.default_rng(1)
fig, axes = plt.subplots(1, len(NEG_KNOBS), figsize=(4.2 * len(NEG_KNOBS), 3.8), squeeze=False)
for ax, k in zip(axes[0], NEG_KNOBS):
    has_neg = (U[k] < 0).any(axis=1)
    varied = np.flatnonzero(~has_neg & (np.abs(U[k] - 1).max(axis=1) > 1e-6))
    for idx in rng.choice(varied, size=min(40, len(varied)), replace=False):
        ax.plot(SIGMAS, U[k][idx], color="0.7", lw=0.8)
    for idx in rng.choice(np.flatnonzero(has_neg), size=min(40, has_neg.sum()), replace=False):
        ax.plot(SIGMAS, U[k][idx], color="C3", lw=0.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_title(k.replace("_UBGenie", "") + f"\n{has_neg.mean():.2%} of events have a negative point", fontsize=9)
    ax.set_xlabel("knob value [σ]")
axes[0][0].set_ylabel("event weight")
axes[0][0].plot([], [], color="C3", label="has a negative point")
axes[0][0].plot([], [], color="0.7", label="no negative point")
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(NEG_KNOBS), figsize=(4.2 * len(NEG_KNOBS), 3.6), squeeze=False)
for ax, k in zip(axes[0], NEG_KNOBS):
    bins = np.linspace(min(-1.0, U[k].min()), max(4.0, np.percentile(U[k], 99.99)), 120)
    for j, color in zip([0, 1, 5, 6], ["C0", "C9", "C1", "C3"]):
        ax.hist(U[k][:, j], bins=bins, histtype="step", color=color, label=f"{SIGMAS[j]:+d}σ")
    ax.axvspan(bins[0], 0, color="0.9", zorder=-1)
    ax.set_yscale("log")
    ax.set_title(k.replace("_UBGenie", ""), fontsize=9)
    ax.set_xlabel("event weight")
axes[0][0].set_ylabel("events")
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Which events get negative weights?

Fraction of events with a negative weight anywhere in the curve, split by true interaction properties and by
the BDT reco category (the analysis channel).  `wc_truth_nuScatType` is the GENIE scattering code
(1 QE, 3 DIS, 4 RES, 5 COH, 10 MEC, as in `signal_categories.py`).

In [ ]:
neg_any = {k: (U[k] < 0).any(axis=1) for k in NEG_KNOBS}
ev_neg = ev.select(["filetype", "reco_category"] + TRUTH_COLS).with_columns(
    [pl.Series(f"neg_{k}", neg_any[k]) for k in NEG_KNOBS])
groupings = [("wc_truth_nuScatType", None), ("wc_truth_isCC", None), ("wc_truth_NprimPio", 3),
             ("wc_true_num_prim_protons", 4), ("wc_true_num_prim_neutrons", 4), ("filetype", None)]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, (col, cap) in zip(axes.flat, groupings):
    d = ev_neg
    if cap is not None:
        d = d.with_columns(pl.col(col).clip(upper_bound=cap))
    g = d.group_by(col).agg([pl.len().alias("n")] + [pl.col(f"neg_{k}").mean() for k in NEG_KNOBS]).sort(col).filter(pl.col("n") > 200)
    labels = [str(v) + ("+" if cap is not None and v == cap else "") for v in g[col].to_list()]
    x = np.arange(len(labels))
    width = 0.8 / len(NEG_KNOBS)
    for i, k in enumerate(NEG_KNOBS):
        ax.bar(x + i * width, g[f"neg_{k}"].to_numpy(), width=width, label=k.replace("_UBGenie", ""))
    ax.set_xticks(x + 0.4 - width / 2, labels, rotation=30 if col == "filetype" else 0, ha="right" if col == "filetype" else "center", fontsize=8)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("fraction with a negative weight")
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Does it matter for PROfit?

The analysis bins are read from the PROfit XML: each overlay subchannel's `<branch>` picks one `reco_category`,
filled in `wc_kine_reco_Enu` with that channel's bin edges.  For every knob, channel, bin and σ point we compute the
PROfit response $R_\sigma / R_0$ twice: as PROfit does, and with the negative event weights clipped at zero.  The
difference between the two is the effect of the negative weights.

In [ ]:
xml_text = re.sub(r"<\?xml[^>]*\?>", "", open(XML_PATH).read()).replace("&&", "&amp;&amp;")
root = ET.fromstring(f"<root>{xml_text}</root>")
def _edges(bins):
    """A channel's bin edges: explicit edges="...", or min/max/nbins."""
    if bins.get("edges") is not None:
        return np.array(bins.get("edges").split(), dtype=float)
    return np.linspace(float(bins.get("min")), float(bins.get("max")), int(bins.get("nbins")) + 1)
channel_edges = {c.get("name"): _edges(c.find("bins")) for c in root.findall("channel")}
channel_category = {}
for b in root.iter("branch"):
    m = re.match(r"nu_uBooNE_(.+)_overlays$", b.get("associated_subchannel"))
    cat = re.search(r"reco_category==(\d+)", b.get("weight_1"))
    if m and cat:
        channel_category[m.group(1)] = int(cat.group(1))
print(f"{len(channel_category)} overlay channels:", channel_category)

reco_cat = ev["reco_category"].to_numpy()
enu = ev["wc_kine_reco_Enu"].to_numpy()

def bin_responses(u, clip_negative=False):
    """{channel: (n_bins, 7) array of PROfit responses R_sigma / R_0}, plus per-bin weight sums."""
    if clip_negative:
        u = np.clip(u, 0, None)
    out, sumw = {}, {}
    for ch, cat in channel_category.items():
        edges = channel_edges[ch]
        sel = reco_cat == cat
        idx = np.digitize(enu[sel], edges) - 1
        ok = (idx >= 0) & (idx < len(edges) - 1)
        nb = len(edges) - 1
        cv = np.bincount(idx[ok], weights=w[sel][ok], minlength=nb)
        var = np.vstack([np.bincount(idx[ok], weights=(w[sel] * u[sel, j])[ok], minlength=nb) for j in range(7)]).T
        with np.errstate(invalid="ignore", divide="ignore"):
            r = var / cv[:, None]
            out[ch] = r / r[:, [CV_INDEX]]
        sumw[ch] = cv
    return out, sumw

resp, sumw = {}, None
for k in NEG_KNOBS:
    resp[k] = {"profit": bin_responses(U[k])[0], "clipped": bin_responses(U[k], clip_negative=True)[0]}
sumw = bin_responses(U[NEG_KNOBS[0]])[1]

summary = []
for k in NEG_KNOBS:
    for j, s in enumerate(SIGMAS):
        if s == 0:
            continue
        shifts = np.concatenate([resp[k]["profit"][ch][:, j] - 1 for ch in channel_category])
        diffs = np.concatenate([resp[k]["profit"][ch][:, j] - resp[k]["clipped"][ch][:, j] for ch in channel_category])
        mins = np.concatenate([resp[k]["profit"][ch][:, j] for ch in channel_category])
        summary.append(dict(knob=k.replace("_UBGenie", ""), sigma=int(s),
                            max_abs_shift=np.nanmax(np.abs(shifts)),
                            max_abs_effect_of_negatives=np.nanmax(np.abs(diffs)),
                            n_bins_effect_over_1pct=int(np.sum(np.abs(diffs) > 0.01)),
                            min_response=np.nanmin(mins)))
summary = pl.DataFrame(summary)
with pl.Config(tbl_rows=40, float_precision=4):
    print(summary)

Response per bin at ±1σ and ±3σ (solid: as PROfit computes it; dashed: negatives clipped at zero) for the channels
where the negative weights move the response the most, with the bin's summed weight underneath.

In [ ]:
for k in NEG_KNOBS:
    effect = {ch: np.nanmax(np.abs(resp[k]["profit"][ch] - resp[k]["clipped"][ch])) for ch in channel_category}
    top = sorted(effect, key=lambda ch: -np.nan_to_num(effect[ch]))[:4]
    fig, axes = plt.subplots(2, 4, figsize=(17, 5.5), sharex="col", gridspec_kw={"height_ratios": [2.2, 1]})
    for col, ch in enumerate(top):
        edges = channel_edges[ch]
        centers = 0.5 * (edges[1:] + edges[:-1])
        ax = axes[0][col]
        for j, color in [(2, "C0"), (4, "C1"), (0, "C9"), (6, "C3")]:
            ax.step(edges, np.append(resp[k]["profit"][ch][:, j], resp[k]["profit"][ch][-1, j]), where="post", color=color, label=f"{SIGMAS[j]:+d}σ")
            ax.step(edges, np.append(resp[k]["clipped"][ch][:, j], resp[k]["clipped"][ch][-1, j]), where="post", color=color, ls="--", lw=0.9)
        ax.axhline(1, color="k", lw=0.6)
        ax.set_title(f"{ch}  (max effect {effect[ch]:.3f})", fontsize=9)
        axes[1][col].bar(centers, sumw[ch], width=np.diff(edges), color="0.6")
        axes[1][col].set_xlabel("reco Enu [MeV]")
    axes[0][0].set_ylabel("PROfit response R/R0")
    axes[1][0].set_ylabel("sum of weights")
    axes[0][0].legend(fontsize=7, ncol=2)
    fig.suptitle(k.replace("_UBGenie", ""), y=1.0)
    plt.tight_layout()
    plt.show()

Effect of the negative weights versus how populated the bin is, at ±3σ where they occur most.  Bins where the negative weights have no effect at all are drawn at the 10⁻⁶ floor.

In [ ]:
fig, axes = plt.subplots(1, len(NEG_KNOBS), figsize=(4.2 * len(NEG_KNOBS), 3.6), squeeze=False, sharey=True)
for ax, k in zip(axes[0], NEG_KNOBS):
    for j, color in [(0, "C9"), (6, "C3")]:
        x = np.concatenate([sumw[ch] for ch in channel_category])
        y = np.concatenate([np.abs(resp[k]["profit"][ch][:, j] - resp[k]["clipped"][ch][:, j]) for ch in channel_category])
        ax.scatter(x, np.maximum(y, 1e-6), s=8, color=color, label=f"{SIGMAS[j]:+d}σ")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("bin sum of weights")
    ax.set_title(k.replace("_UBGenie", ""), fontsize=9)
axes[0][0].set_ylabel("|response(PROfit) − response(clipped)|")
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Conclusions

* **Which knobs.**  Only the hA fate-fraction knobs FrCEx_N (8.5% of events with a negative point), FrInel_pi (0.9%),
  FrCEx_pi (0.8%) and FrAbs_pi (0.1%) produce negative weights for valid-tune events; the other 51 UBGenie knobs,
  including the nucleon FrAbs/FrInel and the MFP knobs, never do, and none are non-finite.
* **Where in σ.**  Always on the + side: at +3σ (6.2% of events for FrCEx_N, 0.8% for FrInel_pi and FrCEx_pi) and at
  +2σ (2% for FrCEx_N, < 0.05% for the others).  At ±1σ there are none except a handful of FrAbs_pi events at +1σ.
* **What the curves show.**  Events whose hadron underwent the varied fate get a weight rising with σ, and on the − side
  it is clamped at zero (the curves are flat between −3σ and −2σ for the CEx knobs, where that fate fraction has reached
  zero).  Events whose hadrons underwent the other fates get a compensating weight that falls roughly linearly with σ
  and is *not* clamped, so it crosses zero around +2σ to +3σ.  This is consistent with the + side extrapolating the
  other fates' fractions past zero, i.e. the outer + points are outside the range where this reweighting is physical.
* **Which events.**  The FrCEx_N negatives are concentrated in nucleon-rich final states: ~16% of MEC events and ~17%
  of events with 2–3 primary protons, versus ~8% for QE/RES and ~4% for DIS.
* **Impact on PROfit.**  Recomputing PROfit's per-bin response with the negative weights clipped at zero:
  * FrCEx_N at +3σ: the negatives change the response by more than 1% in 216 bins, up to 0.26 in a sparse 1g0p1mu
    bin.  The effect is not only a sparse-bin one: it grows with the bin population to ~3–5% in the most populated
    bins, because the falling weights of many events cross zero together.
  * FrCEx_N at +2σ: up to 0.023 (9 bins above 1%).
  * FrAbs_pi, FrCEx_pi, FrInel_pi: below 0.01 everywhere, at every σ.
  * ±1σ: no effect beyond 0.004 for any knob.

  Clipping at zero is only a comparison point, not a correct alternative: it stops the probabilities going negative
  but no longer keeps the fate fractions summing to one.
* **Does it matter?**  These are `type="spline"` fit parameters, so the outer + points mainly enter if the fit pulls
  FrCEx_N above +1σ, where the spline is interpolated toward the +2σ and +3σ points (depending on PROfit's spline
  construction, those knots may also shape the segment just below +1σ; not checked here).  Worth checking the FrCEx_N
  pull in the fits.  If it is large, options would be restricting its fit range (PROfit's `restrict` attribute) or treating
  the +2σ/+3σ points differently when writing the PROfit inputs.
